> **Superseded.** This notebook was the local-continuation half of the single-question walkthrough (Stages 4-7), from when Stella could not be embedded on this machine and query vectors had to be hand-carried from Colab one file at a time. Now that the pipeline runs against a Colab-backed kernel directly from VS Code, the full batch pipeline (indexing + query embedding + retrieval/generation for all 150 questions) lives in `vector_rag_pipeline.ipynb` instead. Kept here for reference on how the single-question version worked.

# Vector RAG Pipeline — Local Continuation (Stage 4 onward)

**Companion to** `vector_rag_pipeline.ipynb` (which runs in Colab, where Stella 1.5B can actually load).

## Why this notebook exists

Stella 1.5B (~6GB) can't load on this machine — confirmed by trying: `SentenceTransformer(...)` itself fails, not just batch-encoding chunks. So anything that needs to turn text into a dense vector — chunks *or* queries — has to happen in Colab.

Everything downstream of embedding doesn't need Stella at all:
- Query expansion → an LLM API call (Gemini)
- Hybrid retrieval → numpy (dense) + BM25 (pure Python), both already computed/cheap
- Reranking → Voyage API call
- Selection + generation → LLM API calls

So the split is: **Colab embeds, local does everything else.**

## What you need before running this

1. `experiments/results/vector_rag_index/{DOC_NAME}_chunks.parquet` and `_dense.npy` — downloaded from the Colab notebook's Stage 3 (`FFYZxAIVfXpE` cell saves these).
2. A query vector for the question you're testing — from the Colab notebook's new **Stage 3b** query-embedding utility cell. You'll run Stage 4 below first to get the expanded query text, paste it into that Colab cell, download the result, and come back.
3. `.env` filled in with `GOOGLE_API_KEY` and `VOYAGE_API_KEY` (copied from `.env.example` — already scaffolded, just needs real keys).

## Pipeline stages covered here

```
④ Query expansion    ── LLM rewrites the question for better retrieval matching
⑤ Hybrid retrieval   ── 0.85 × dense + 0.15 × sparse(BM25), top-20
⑥ Rerank             ── Voyage rerank-2, top-20 → top-10
⑦ Generate           ── selection agent filters chunks, then Gemini answers
```


---
## Stage 0 — Setup

Local paths, `.env`, and cost tracking. No Drive mount, no git clone — we're already inside the repo.

In [1]:
import os, sys, json, time, textwrap, re
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Walk up from the notebook's cwd until we find the repo root (has pyproject.toml).
# Works regardless of whether Jupyter was launched from the repo root or from
# pipelines/vector_rag/.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Could not find repo root (no pyproject.toml found walking up from cwd)")

sys.path.insert(0, str(REPO_ROOT))  # so `from evaluation.cost_tracker import CostTracker` works

DATA_DIR  = REPO_ROOT / "data"
PDF_DIR   = REPO_ROOT / "pdfs"
INDEX_DIR = REPO_ROOT / "experiments" / "results" / "vector_rag_index"
QUERIES_DIR = INDEX_DIR / "queries"

# override=True: without it, re-running this cell after editing .env keeps the
# stale value already sitting in the kernel's os.environ from an earlier run
# in this session, instead of picking up your edit.
load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
VOYAGE_API_KEY = os.getenv("VOYAGE_API_KEY", "")
GROQ_API_KEY   = os.getenv("GROQ_API_KEY", "")  # LLM judge — Llama 3.3 70B via Groq's free tier

print(f"REPO_ROOT: {REPO_ROOT}")
for name, val in [
    ("GOOGLE_API_KEY", GOOGLE_API_KEY),
    ("VOYAGE_API_KEY", VOYAGE_API_KEY),
    ("GROQ_API_KEY", GROQ_API_KEY),
]:
    print(f"  {name:<18}: {'ok' if val else 'MISSING - fill in .env'}")

REPO_ROOT: /Users/shaliqshukoor/dev/thesis-project
  GOOGLE_API_KEY    : ok
  VOYAGE_API_KEY    : ok
  GROQ_API_KEY      : ok


In [2]:
from evaluation.cost_tracker import CostTracker

cost_tracker = CostTracker(REPO_ROOT / "experiments" / "results" / "vector_rag_costs.jsonl")
print(f"Logging costs to {cost_tracker.log_path}")
print("Note: PRICING_PER_MILLION_TOKENS in evaluation/cost_tracker.py is still unfilled (all None),")
print("so cost_usd will log as None for now. Token counts are logged regardless — fill in the")
print("published rates before the real experiment runs so cost numbers become meaningful.")

from groq import Groq

# Llama 3.3 70B was the original pick but Groq had fully deprecated it (404 on this
# account) — gpt-oss-120b (OpenAI open-weight, hosted on Groq) is what's actually live
# on the free tier. Still a different vendor family from Gemini/DeepSeek, so it still
# satisfies the self-enhancement-bias rationale. Verified live against all three label
# categories (Correct/Incorrect/Failure to Answer) before adopting.
JUDGE_MODEL = "openai/gpt-oss-120b"
judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
if judge_client is None:
    print("\n⚠ GROQ_API_KEY not set — the answer_scorer LLM-judge fallback will raise if a "
          "question can't be scored deterministically (i.e. no extractable number on one/both sides, "
          "or a purely descriptive question). Fine for numeric lookup questions like this one; fill "
          "in the key before testing descriptive/multi-hop questions.")

Logging costs to /Users/shaliqshukoor/dev/thesis-project/experiments/results/vector_rag_costs.jsonl
Note: PRICING_PER_MILLION_TOKENS in evaluation/cost_tracker.py is still unfilled (all None),
so cost_usd will log as None for now. Token counts are logged regardless — fill in the
published rates before the real experiment runs so cost numbers become meaningful.


---
## Restore: pick the question, load the saved index

This rebuilds the same `row` / `TEST_QUESTION` / etc. that Stage 1 in the Colab notebook produced for `TEST_IDX = 0`, then loads the chunks + dense embeddings you downloaded, and rebuilds the BM25 index (instant, no need to save/load it).

In [3]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

TEST_IDX = 0   # reverted to 0: this is the question we have a query vector for already
row = df.iloc[TEST_IDX]

TEST_DOC_NAME       = row.doc_name
TEST_QUESTION       = row.question
TEST_ANSWER         = row.answer
TEST_EVIDENCE_PAGES = [e["evidence_page_num"] for e in row.evidence]
FB_ID               = row.financebench_id

print(f"financebench_id : {FB_ID}")
print(f"Document        : {TEST_DOC_NAME}")
print(f"Question        : {TEST_QUESTION}")
print(f"Gold answer     : {TEST_ANSWER}")
print(f"Evidence pages  : {TEST_EVIDENCE_PAGES}  (0-indexed)")

financebench_id : financebench_id_03029
Document        : 3M_2018_10K
Question        : What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Gold answer     : $1577.00
Evidence pages  : [59]  (0-indexed)


In [4]:
chunks_path = INDEX_DIR / f"{TEST_DOC_NAME}_chunks.parquet"
dense_path  = INDEX_DIR / f"{TEST_DOC_NAME}_dense.npy"

if not chunks_path.exists() or not dense_path.exists():
    raise FileNotFoundError(
        f"Missing {chunks_path.name} and/or {dense_path.name} in {INDEX_DIR}.\n"
        f"Download them from the Colab notebook's Stage 3 (cell FFYZxAIVfXpE saves them to "
        f"the same filenames under vector_rag_index/ in Drive) and drop them here."
    )

chunks_df        = pd.read_parquet(chunks_path)
dense_embeddings = np.load(dense_path)
assert len(chunks_df) == dense_embeddings.shape[0], "chunks_df and dense_embeddings are out of sync"

print(f"Loaded {len(chunks_df)} chunks, dense_embeddings shape {dense_embeddings.shape}")

Loaded 375 chunks, dense_embeddings shape (375, 1024)


In [5]:
from rank_bm25 import BM25Okapi

def bm25_tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9$][a-z0-9.,%$-]*", text.lower())

def format_query(query: str) -> str:
    return f"Instruct: Given a web search query, retrieve relevant passages that answer the query.\nQuery: {query}"

tokenized_corpus = [bm25_tokenize(t) for t in chunks_df["text"]]
bm25_index = BM25Okapi(tokenized_corpus)
print(f"BM25 index rebuilt over {len(tokenized_corpus)} chunks")

BM25 index rebuilt over 375 chunks


---
## Stage 4 — Query expansion

Ask an LLM to rewrite the question so it better matches how the answer is likely *phrased* in the filing — e.g. a question about "capital expenditure" should also match text saying "Purchases of property, plant and equipment (PP&E)". We use Gemini 2.5 Flash (same generation model as the rest of the pipeline).

In [6]:
from google import genai

genai_client = genai.Client(api_key=GOOGLE_API_KEY)

GENERATION_MODEL = "gemini-3.5-flash"  # gemini-2.5-flash was deprecated for new API keys/projects

EXPANSION_PROMPT = """You are helping a retrieval system find the right passage in a financial \
filing (10-K/10-Q). Rewrite the question below into a short expanded search query: include likely \
synonyms, exact financial line-item names, and related terms that would appear near the answer in \
the filing. Return ONLY the expanded query text - no preamble, no explanation, no quotes.

Question: {question}"""

def expand_query(question: str, model: str = GENERATION_MODEL):
    resp = genai_client.models.generate_content(
        model=model,
        contents=EXPANSION_PROMPT.format(question=question),
    )
    return resp.text.strip(), resp.usage_metadata

expanded_query, exp_usage = expand_query(TEST_QUESTION)

cost_tracker.log(
    pipeline="vector_rag", stage="query_expansion", model=GENERATION_MODEL,
    input_tokens=exp_usage.prompt_token_count, output_tokens=exp_usage.candidates_token_count,
    doc_name=TEST_DOC_NAME, financebench_id=FB_ID,
)

print(f"Original question:\n  {TEST_QUESTION}")
print(f"\nExpanded query:\n  {expanded_query}")

Original question:
  What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

Expanded query:
  3M Company Consolidated Statements of Cash Flows FY2018 2018 purchases of property plant and equipment capital expenditures additions PP&E investing activities millions


### ⏸ Round-trip to Colab

Copy the `expanded_query` printed above. In the **Colab** notebook's new **Stage 3b** cell:

```python
QUERY_TEXT = "<paste expanded_query here>"
QUERY_TAG  = "expanded"
FB_ID      = <the FB_ID printed above, as a string>
```

Run it, download `{FB_ID}__expanded.npy` and `{FB_ID}__expanded.json` from `vector_rag_index/queries/` in Drive, and drop both into `experiments/results/vector_rag_index/queries/` in this repo. Then run the cell below.

In [7]:
QUERIES_DIR

PosixPath('/Users/shaliqshukoor/dev/thesis-project/experiments/results/vector_rag_index/queries')

In [8]:
query_vec_path = QUERIES_DIR/ f"{FB_ID}__expanded.npy"

if not query_vec_path.exists():
    raise FileNotFoundError(
        f"{query_vec_path.name} not found in {QUERIES_DIR}.\n"
        f"Go embed it in Colab first (Stage 3b) - see the markdown cell above for exact values to use:\n"
        f"  QUERY_TEXT = {expanded_query!r}\n  QUERY_TAG  = 'expanded'\n  FB_ID      = {FB_ID!r}"
    )

query_vec = np.load(query_vec_path)
print(f"Loaded query vector {query_vec_path.name}  shape={query_vec.shape}")

Loaded query vector financebench_id_03029__expanded.npy  shape=(1024,)


---
## Stage 5 — Hybrid retrieval

Combine dense (cosine similarity) and sparse (BM25) scores: `hybrid = 0.85 × dense + 0.15 × sparse`, per the paper.

**Design note on normalization:** dense cosine similarity is already in `[-1, 1]` (usually `[0, 1]` for normalized embeddings), but raw BM25 scores are unbounded term-frequency sums that can be in the tens or hundreds. Averaging the two directly with an 0.85/0.15 weighting would let dense dominate almost completely regardless of the BM25 weight, since the two scores live on totally different scales. We min-max normalize BM25 scores into `[0, 1]` over the candidate set before combining, so the 0.85/0.15 weights actually mean what they say.

In [9]:
from evaluation.retrieval_metrics import compute_retrieval_metrics

ALPHA = 0.85          # dense weight, per the paper

# TEMPORARY for this smoke test: paper spec is top-20. The full top-20 set from
# this doc totals ~10.3K tokens, over Voyage's free-tier cap of 10K tokens/minute,
# so Stage 6 fails on every call. Dropping to top-10 (~5.7K tokens, per our
# token_count column) leaves comfortable headroom under the cap even if Voyage's
# own tokenizer counts slightly differently than ours. REVERT TO 20 once a Voyage
# payment method is added (needed anyway for the full 150-question run, where
# 20-chunk reranks are unavoidable at scale). Note this also caps what Recall@15/
# MRR@15 below can measure — with only 10 candidates, k=15 can't show anything
# beyond what k=10 already shows.
TOP_K_HYBRID = 10

dense_scores    = dense_embeddings @ query_vec
bm25_scores_raw = bm25_index.get_scores(bm25_tokenize(expanded_query))

bm25_min, bm25_max = bm25_scores_raw.min(), bm25_scores_raw.max()
bm25_scores_norm = (bm25_scores_raw - bm25_min) / (bm25_max - bm25_min + 1e-9)

hybrid_scores = ALPHA * dense_scores + (1 - ALPHA) * bm25_scores_norm

top20_idx = np.argsort(-hybrid_scores)[:TOP_K_HYBRID]
top20 = chunks_df.iloc[top20_idx].copy()
top20["hybrid_score"] = hybrid_scores[top20_idx]
top20["dense_score"]  = dense_scores[top20_idx]
top20["bm25_score"]   = bm25_scores_raw[top20_idx]

hybrid_metrics = compute_retrieval_metrics(
    ranked_pages=top20["page_num"].tolist(),
    gold_pages=TEST_EVIDENCE_PAGES,
    financebench_id=FB_ID,
    doc_name=TEST_DOC_NAME,
    stage="hybrid_top10",
)

print(f"Gold evidence page(s): {TEST_EVIDENCE_PAGES}")
print(f"Recall@k: {hybrid_metrics.recall_at_k}")
print(f"MRR@k:    {hybrid_metrics.mrr_at_k}")
print(f"Total tokens in candidate set: {chunks_df.loc[top20.index, 'token_count'].sum()}")
top20[["chunk_id", "page_num", "hybrid_score", "dense_score", "bm25_score"]]

Gold evidence page(s): [59]
Recall@k: {5: 1.0, 10: 1.0, 15: 1.0}
MRR@k:    {5: 0.5, 10: 0.5, 15: 0.5}
Total tokens in candidate set: 5666


,chunk_id,page_num,hybrid_score,dense_score,bm25_score
112,3M_2018_10K__p0045_c02,45,0.798255,0.762653,52.888848
138,3M_2018_10K__p0059_c00,59,0.791608,0.769142,48.600285
121,3M_2018_10K__p0048_c01,48,0.771815,0.746725,48.340150
113,3M_2018_10K__p0046_c00,46,0.713504,0.763936,22.621813
198,3M_2018_10K__p0082_c01,82,0.677604,0.695541,30.461800
136,3M_2018_10K__p0057_c00,57,0.673847,0.706364,25.893418
114,3M_2018_10K__p0046_c01,46,0.668433,0.708202,23.433753
120,3M_2018_10K__p0048_c00,48,0.667134,0.684286,30.143410
110,3M_2018_10K__p0045_c00,45,0.655384,0.670293,30.194132
310,3M_2018_10K__p0126_c01,126,0.654354,0.694791,22.488926


---
## Stage 6 — Rerank (Voyage rerank-2)

Re-score all 20 chunk-question pairs with a dedicated reranking model and keep the top 10. Rerankers see the query and passage together (cross-encoder), which is more accurate than the independent dense/sparse scores above, but too slow to run over the whole corpus — hence retrieve-then-rerank.

In [10]:
import voyageai
from evaluation.retrieval_metrics import compute_retrieval_metrics

voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)

TOP_K_RERANK = 10

rerank_result = voyage_client.rerank(
    query=expanded_query,
    documents=top20["text"].tolist(),
    model="rerank-2",
    top_k=TOP_K_RERANK,
)

reranked_order  = [r.index for r in rerank_result.results]
reranked_scores = [r.relevance_score for r in rerank_result.results]

top10 = top20.iloc[reranked_order].copy()
top10["rerank_score"] = reranked_scores

cost_tracker.log(
    pipeline="vector_rag", stage="rerank", model="voyage-rerank-2",
    input_tokens=rerank_result.total_tokens, output_tokens=0,
    doc_name=TEST_DOC_NAME, financebench_id=FB_ID,
)

rerank_metrics = compute_retrieval_metrics(
    ranked_pages=top10["page_num"].tolist(),
    gold_pages=TEST_EVIDENCE_PAGES,
    financebench_id=FB_ID,
    doc_name=TEST_DOC_NAME,
    stage="rerank_top10",
)

print(f"Recall@k: {rerank_metrics.recall_at_k}")
print(f"MRR@k:    {rerank_metrics.mrr_at_k}")
top10[["chunk_id", "page_num", "rerank_score"]]

/Users/shaliqshukoor/dev/thesis-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Recall@k: {5: 1.0, 10: 1.0, 15: 1.0}
MRR@k:    {5: 1.0, 10: 1.0, 15: 1.0}


,chunk_id,page_num,rerank_score
138,3M_2018_10K__p0059_c00,59,0.875000
112,3M_2018_10K__p0045_c02,45,0.789062
121,3M_2018_10K__p0048_c01,48,0.781250
310,3M_2018_10K__p0126_c01,126,0.609375
113,3M_2018_10K__p0046_c00,46,0.605469
198,3M_2018_10K__p0082_c01,82,0.558594
120,3M_2018_10K__p0048_c00,48,0.550781
136,3M_2018_10K__p0057_c00,57,0.550781
110,3M_2018_10K__p0045_c00,45,0.507812
114,3M_2018_10K__p0046_c01,46,0.406250


---
## Stage 7 — Selection agent + generation

An LLM filters the 10 reranked chunks down to only the ones actually needed (drops redundant/irrelevant passages that made it through retrieval on keyword overlap alone), then a second call generates the final answer from just the selected chunks.

In [11]:
def format_passages(df: pd.DataFrame) -> str:
    return "\n\n".join(
        f"[{r.chunk_id}] (page {r.page_num})\n{r.text}" for _, r in df.iterrows()
    )

SELECTION_PROMPT = """You are filtering retrieved passages from a financial filing before they are \
passed to an answer-generation model. Given the question and candidate passages below, return ONLY \
the passage IDs that are actually needed to answer the question, as a JSON array of strings \
(e.g. ["id1", "id2"]). Drop passages that are irrelevant or redundant. Keep at least one passage. \
Return nothing except the JSON array.

Question: {question}

Candidate passages:
{passages}"""

def select_chunks(question: str, candidates_df: pd.DataFrame, model: str = GENERATION_MODEL):
    resp = genai_client.models.generate_content(
        model=model,
        contents=SELECTION_PROMPT.format(question=question, passages=format_passages(candidates_df)),
    )
    raw = resp.text.strip()
    match = re.search(r"\[.*\]", raw, re.DOTALL)
    ids = json.loads(match.group(0) if match else raw)
    return ids, resp.usage_metadata

selected_ids, sel_usage = select_chunks(TEST_QUESTION, top10)
selected_df = top10[top10.chunk_id.isin(selected_ids)]
print(f"Selected {len(selected_df)}/{len(top10)} chunks: {selected_ids}")


Selected 1/10 chunks: ['3M_2018_10K__p0059_c00']


In [12]:
from evaluation.answer_scorer import score_answer

GENERATION_PROMPT = """You are a financial analyst answering a question using only the excerpts \
below from a company's SEC filing. Answer concisely and precisely, matching the format the question \
expects (a number, a yes/no with brief reasoning, etc). If the excerpts don't contain enough \
information to answer, say so explicitly rather than guessing.

Question: {question}

Excerpts:
{passages}

Answer:"""

def generate_answer(question: str, context_df: pd.DataFrame, model: str = GENERATION_MODEL):
    resp = genai_client.models.generate_content(
        model=model,
        contents=GENERATION_PROMPT.format(question=question, passages=format_passages(context_df)),
    )
    return resp.text.strip(), resp.usage_metadata

answer, gen_usage = generate_answer(TEST_QUESTION, selected_df)

cost_tracker.log(
    pipeline="vector_rag", stage="generation", model=GENERATION_MODEL,
    input_tokens=gen_usage.prompt_token_count, output_tokens=gen_usage.candidates_token_count,
    doc_name=TEST_DOC_NAME, financebench_id=FB_ID,
)

score_result, judge_usage = score_answer(
    TEST_QUESTION, TEST_ANSWER, answer, judge_client=judge_client, judge_model=JUDGE_MODEL
)
if judge_usage is not None:
    cost_tracker.log(
        pipeline="vector_rag", stage="judge", model=JUDGE_MODEL,
        input_tokens=judge_usage["input_tokens"], output_tokens=judge_usage["output_tokens"],
        doc_name=TEST_DOC_NAME, financebench_id=FB_ID,
    )

print("=" * 70)
print(f"Question    : {TEST_QUESTION}")
print(f"Gold answer : {TEST_ANSWER}")
print(f"Model answer: {answer}")
print("-" * 70)
print(f"Score       : {score_result.label}  (method={score_result.method})")
if score_result.method == "deterministic":
    print(f"  gold_value={score_result.gold_value}  matched_value={score_result.matched_value}")
else:
    print(f"  reasoning: {score_result.reasoning}")
print("=" * 70)

Question    : What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Gold answer : $1577.00
Model answer: Based on the Consolidated Statement of Cash Flows, 3M's capital expenditures (represented by "Purchases of property, plant and equipment (PP&E)") for FY2018 were **$1,577 million** (recorded as an outflow of $1,577 million).
----------------------------------------------------------------------
Score       : Correct  (method=deterministic)
  gold_value=1577000000.0  matched_value=1577000000.0


In [13]:

cost_tracker.log(
    pipeline="vector_rag", stage="selection", model=GENERATION_MODEL,
    input_tokens=sel_usage.prompt_token_count, output_tokens=sel_usage.candidates_token_count,
    doc_name=TEST_DOC_NAME, financebench_id=FB_ID,
)

print(f"Selected {len(selected_df)}/{len(top10)} chunks: {selected_ids}")
selected_df[["chunk_id", "page_num", "rerank_score"]]

Selected 1/10 chunks: ['3M_2018_10K__p0059_c00']


,chunk_id,page_num,rerank_score
138,3M_2018_10K__p0059_c00,59,0.875


---
## ✋ End of Stage 7 — full pipeline runs locally for one question

We now have an end-to-end run: expand → hybrid retrieve → rerank → select → generate, all locally except the one Stella round-trip through Colab.

### Next steps
1. **Sanity-check this answer** against the gold answer above, and manually skim `selected_df` — does the pipeline actually use the right evidence, or get lucky?
2. Once this looks right, embed the **full 84-document corpus** in Colab (Stage 3, looped over every `doc_name` instead of just the one test document) and download all the `*_chunks.parquet` / `*_dense.npy` files.
3. Batch-embed all 150 **questions** in Colab in one pass too (extend Stage 3b into a loop), instead of the one-at-a-time round-trip used here — this is what makes the full run practical.
4. Wrap Stages 4–7 in this notebook into a function (`run_vector_rag(financebench_id) -> answer, cost, latency`) so it can be called in a loop over all 150 questions × both generation models, per the timeline's Week 8–9 milestone.
5. Wire in `evaluation/retrieval_metrics.py` (MRR@k, Recall@k against `TEST_EVIDENCE_PAGES`) and `evaluation/answer_scorer.py` once those exist, instead of the manual gold-page checks used above.

Also remember to fill in `PRICING_PER_MILLION_TOKENS` in `evaluation/cost_tracker.py` with current published rates before the real experiment run — costs have been logged as token counts throughout, but `cost_usd` is `None` until those prices are filled in.